# 📊 RETO 6 - OPEN DATA: Preparación e Integración de Datasets

**Proyecto:** Observatorio de Comercio Minorista - España  
**Objetivo:** Enriquecer dataset principal con Open Data INE  
**Autor:** Ana BM  
**Fecha:** Marzo 2026  

---

## 🎯 Objetivos de este Notebook:

1. ✅ Cargar dataset principal + archivos Open Data
2. ✅ Aplicar mapeos geográficos (Istanbul → España)
3. ✅ Aplicar mapeos de categorías (Dataset → ECOICOP)
4. ✅ Preparar datasets para integración (tipos de datos, fechas)
5. ✅ **INTEGRACIÓN:** LEFT JOIN dataset principal + IPC
6. ✅ Validar dataset enriquecido final
7. ✅ Guardar dataset enriquecido para análisis

---

## 📝 METODOLOGÍA:

**Este es el paso CRÍTICO del proyecto** donde demostramos:
- Integración de múltiples fuentes de datos
- Transformaciones complejas (mapeos geográficos + categorías)
- Validación de calidad de datos post-integración
- Creación de un dataset enriquecido enterprise-grade

---

## 📦 CELDA 1: Importar Librerías

In [1]:
# Librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Librerías importadas correctamente")
print(f"📅 Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Librerías importadas correctamente
📅 Fecha de ejecución: 2026-03-18 10:26:36


## 🔌 CELDA 2: Montar Google Drive y Definir Rutas

In [2]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Definir rutas
BASE_PATH = '/content/drive/MyDrive/Reto-6-Open-Data/'
RAW_PATH = BASE_PATH + 'data/raw/'
MAPPING_PATH = BASE_PATH + 'data/mapping/'
ENRICHED_PATH = BASE_PATH + 'data/enriched/'

print("\n✅ Google Drive montado correctamente")
print("\n📂 RUTAS CONFIGURADAS:")
print(f"   • Raw data: {RAW_PATH}")
print(f"   • Mapping: {MAPPING_PATH}")
print(f"   • Enriched: {ENRICHED_PATH}")

Mounted at /content/drive

✅ Google Drive montado correctamente

📂 RUTAS CONFIGURADAS:
   • Raw data: /content/drive/MyDrive/Reto-6-Open-Data/data/raw/
   • Mapping: /content/drive/MyDrive/Reto-6-Open-Data/data/mapping/
   • Enriched: /content/drive/MyDrive/Reto-6-Open-Data/data/enriched/


## 📥 CELDA 3: Cargar Todos los Datasets

In [3]:
print("="*80)
print("📥 CARGANDO DATASETS")
print("="*80)

# 1. DATASET PRINCIPAL
print("\n1️⃣ Cargando dataset principal...")
df_principal = pd.read_csv(RAW_PATH + 'customer_shopping_data.csv')
df_principal['invoice_date'] = pd.to_datetime(df_principal['invoice_date'], dayfirst=True)
print(f"   ✅ Dataset principal: {len(df_principal):,} registros")
print(f"   • Período: {df_principal['invoice_date'].min().strftime('%Y-%m-%d')} a {df_principal['invoice_date'].max().strftime('%Y-%m-%d')}")

# 2. OPEN DATA INE
print("\n2️⃣ Cargando Open Data INE...")
df_ipc_nacional = pd.read_csv(RAW_PATH + 'ine_ipc_nacional.csv')
df_ipc_nacional['Fecha'] = pd.to_datetime(df_ipc_nacional['Fecha'])
print(f"   ✅ IPC Nacional: {len(df_ipc_nacional)} registros")

df_ipc_categorias = pd.read_csv(RAW_PATH + 'ine_ipc_categorias.csv')
df_ipc_categorias['Fecha'] = pd.to_datetime(df_ipc_categorias['Fecha'])
print(f"   ✅ IPC Categorías: {len(df_ipc_categorias)} registros")

df_ipc_ccaa = pd.read_csv(RAW_PATH + 'ine_ipc_ccaa.csv')
df_ipc_ccaa['Fecha'] = pd.to_datetime(df_ipc_ccaa['Fecha'])
print(f"   ✅ IPC CCAA: {len(df_ipc_ccaa)} registros")

# 3. ARCHIVOS DE MAPEO
print("\n3️⃣ Cargando archivos de mapeo...")
df_mapeo_geo = pd.read_csv(MAPPING_PATH + 'istanbul_spain_mapping.csv')
print(f"   ✅ Mapeo geográfico: {len(df_mapeo_geo)} malls")

df_mapeo_cat = pd.read_csv(MAPPING_PATH + 'categories_ecoicop_mapping.csv')
print(f"   ✅ Mapeo categorías: {len(df_mapeo_cat)} categorías")

print("\n✅ TODOS LOS DATASETS CARGADOS CORRECTAMENTE")

📥 CARGANDO DATASETS

1️⃣ Cargando dataset principal...
   ✅ Dataset principal: 99,457 registros
   • Período: 2021-01-01 a 2023-03-08

2️⃣ Cargando Open Data INE...
   ✅ IPC Nacional: 36 registros
   ✅ IPC Categorías: 180 registros
   ✅ IPC CCAA: 288 registros

3️⃣ Cargando archivos de mapeo...
   ✅ Mapeo geográfico: 10 malls
   ✅ Mapeo categorías: 8 categorías

✅ TODOS LOS DATASETS CARGADOS CORRECTAMENTE


## 🗺️ CELDA 4: Aplicar Mapeo Geográfico (Istanbul → España)

In [4]:
print("="*80)
print("🗺️ APLICANDO MAPEO GEOGRÁFICO")
print("="*80)

# Crear copia del dataset principal
df_trabajo = df_principal.copy()

print("\n📊 ANTES DEL MAPEO:")
print(f"   • Total registros: {len(df_trabajo):,}")
print(f"   • Shopping malls únicos: {df_trabajo['shopping_mall'].nunique()}")

# Aplicar mapeo geográfico
df_trabajo = df_trabajo.merge(
    df_mapeo_geo[['shopping_mall', 'ciudad_espana', 'ccaa', 'codigo_ccaa']],
    on='shopping_mall',
    how='left'
)

print("\n📊 DESPUÉS DEL MAPEO:")
print(f"   • Total registros: {len(df_trabajo):,}")
print(f"   • Ciudades españolas únicas: {df_trabajo['ciudad_espana'].nunique()}")
print(f"   • CCAA únicas: {df_trabajo['ccaa'].nunique()}")

# Verificar que no haya nulos (todos los malls deben tener mapeo)
nulos_mapeo = df_trabajo['ciudad_espana'].isnull().sum()
if nulos_mapeo > 0:
    print(f"\n⚠️ ADVERTENCIA: {nulos_mapeo} registros sin mapeo geográfico")
else:
    print("\n✅ Mapeo geográfico aplicado correctamente (sin nulos)")

# Mostrar distribución por ciudad
print("\n📊 TOP 5 CIUDADES POR VOLUMEN DE VENTAS:")
print(df_trabajo['ciudad_espana'].value_counts().head())

print("\n✅ MAPEO GEOGRÁFICO COMPLETADO")

🗺️ APLICANDO MAPEO GEOGRÁFICO

📊 ANTES DEL MAPEO:
   • Total registros: 99,457
   • Shopping malls únicos: 10

📊 DESPUÉS DEL MAPEO:
   • Total registros: 99,457
   • Ciudades españolas únicas: 10
   • CCAA únicas: 8

✅ Mapeo geográfico aplicado correctamente (sin nulos)

📊 TOP 5 CIUDADES POR VOLUMEN DE VENTAS:
ciudad_espana
Madrid       19943
Barcelona    19823
Valencia     15011
Sevilla      10161
Bilbao        9781
Name: count, dtype: int64

✅ MAPEO GEOGRÁFICO COMPLETADO


## 🏷️ CELDA 5: Aplicar Mapeo de Categorías (Dataset → ECOICOP)

In [5]:
print("="*80)
print("🏷️ APLICANDO MAPEO DE CATEGORÍAS")
print("="*80)

print("\n📊 ANTES DEL MAPEO:")
print(f"   • Total registros: {len(df_trabajo):,}")
print(f"   • Categorías dataset: {df_trabajo['category'].nunique()}")

# Aplicar mapeo de categorías
df_trabajo = df_trabajo.merge(
    df_mapeo_cat[['category_dataset', 'grupo_ecoicop', 'codigo_ecoicop']],
    left_on='category',
    right_on='category_dataset',
    how='left'
)

# Eliminar columna redundante
df_trabajo = df_trabajo.drop('category_dataset', axis=1)

print("\n📊 DESPUÉS DEL MAPEO:")
print(f"   • Total registros: {len(df_trabajo):,}")
print(f"   • Códigos ECOICOP únicos: {df_trabajo['codigo_ecoicop'].nunique()}")

# Verificar que no haya nulos
nulos_mapeo = df_trabajo['codigo_ecoicop'].isnull().sum()
if nulos_mapeo > 0:
    print(f"\n⚠️ ADVERTENCIA: {nulos_mapeo} registros sin mapeo de categoría")
else:
    print("\n✅ Mapeo de categorías aplicado correctamente (sin nulos)")

# Mostrar distribución
print("\n📊 TOP 5 GRUPOS ECOICOP POR VOLUMEN:")
print(df_trabajo['grupo_ecoicop'].value_counts().head())

print("\n✅ MAPEO DE CATEGORÍAS COMPLETADO")

🏷️ APLICANDO MAPEO DE CATEGORÍAS

📊 ANTES DEL MAPEO:
   • Total registros: 99,457
   • Categorías dataset: 8

📊 DESPUÉS DEL MAPEO:
   • Total registros: 99,457
   • Códigos ECOICOP únicos: 5

✅ Mapeo de categorías aplicado correctamente (sin nulos)

📊 TOP 5 GRUPOS ECOICOP POR VOLUMEN:
grupo_ecoicop
Vestido y calzado                     44521
Otros bienes y servicios              20096
Ocio y cultura                        15068
Alimentos y bebidas no alcohólicas    14776
Comunicaciones                         4996
Name: count, dtype: int64

✅ MAPEO DE CATEGORÍAS COMPLETADO


## 📅 CELDA 6: Preparar Variables Temporales para Integración

In [6]:
print("="*80)
print("📅 PREPARANDO VARIABLES TEMPORALES")
print("="*80)

# Extraer año y mes del dataset principal
df_trabajo['year'] = df_trabajo['invoice_date'].dt.year
df_trabajo['month'] = df_trabajo['invoice_date'].dt.month

# Crear columna de fecha normalizada (primer día del mes) para JOIN
df_trabajo['fecha_mes'] = pd.to_datetime(
    df_trabajo['year'].astype(str) + '-' +
    df_trabajo['month'].astype(str).str.zfill(2) + '-01'
)

print("\n✅ Variables temporales creadas:")
print("   • year (año)")
print("   • month (mes)")
print("   • fecha_mes (fecha normalizada para JOIN)")

print("\n📊 RANGO TEMPORAL DEL DATASET:")
print(f"   • Fecha mínima: {df_trabajo['fecha_mes'].min().strftime('%Y-%m')}")
print(f"   • Fecha máxima: {df_trabajo['fecha_mes'].max().strftime('%Y-%m')}")
print(f"   • Meses únicos: {df_trabajo['fecha_mes'].nunique()}")

# Normalizar fechas en Open Data también
df_ipc_nacional['fecha_mes'] = pd.to_datetime(df_ipc_nacional['Fecha'].dt.to_period('M').dt.to_timestamp())
df_ipc_categorias['fecha_mes'] = pd.to_datetime(df_ipc_categorias['Fecha'].dt.to_period('M').dt.to_timestamp())
df_ipc_ccaa['fecha_mes'] = pd.to_datetime(df_ipc_ccaa['Fecha'].dt.to_period('M').dt.to_timestamp())

print("\n✅ Fechas normalizadas en todos los datasets")

📅 PREPARANDO VARIABLES TEMPORALES

✅ Variables temporales creadas:
   • year (año)
   • month (mes)
   • fecha_mes (fecha normalizada para JOIN)

📊 RANGO TEMPORAL DEL DATASET:
   • Fecha mínima: 2021-01
   • Fecha máxima: 2023-03
   • Meses únicos: 27

✅ Fechas normalizadas en todos los datasets


## 🔗 CELDA 7: INTEGRACIÓN 1 - LEFT JOIN con IPC Nacional

In [7]:
print("="*80)
print("🔗 INTEGRACIÓN 1: IPC NACIONAL")
print("="*80)

print("\n📊 ANTES DE LA INTEGRACIÓN:")
print(f"   • Registros dataset principal: {len(df_trabajo):,}")
print(f"   • Columnas: {len(df_trabajo.columns)}")

# LEFT JOIN con IPC Nacional
df_enriquecido = df_trabajo.merge(
    df_ipc_nacional[['fecha_mes', 'IPC_Nacional']],
    on='fecha_mes',
    how='left'
)

print("\n📊 DESPUÉS DE LA INTEGRACIÓN:")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")
print(f"   • Nueva columna: IPC_Nacional")

# Validar integración
nulos_ipc = df_enriquecido['IPC_Nacional'].isnull().sum()
print(f"\n🔍 VALIDACIÓN:")
print(f"   • Registros con IPC Nacional: {len(df_enriquecido) - nulos_ipc:,} ({(1 - nulos_ipc/len(df_enriquecido))*100:.2f}%)")
print(f"   • Registros sin IPC Nacional: {nulos_ipc:,} ({(nulos_ipc/len(df_enriquecido))*100:.2f}%)")

if nulos_ipc > 0:
    print(f"\n⚠️ Hay {nulos_ipc} registros sin IPC (fechas fuera del rango de Open Data)")
    print("\nFechas sin IPC (primeras 5):")
    fechas_sin_ipc = df_enriquecido[df_enriquecido['IPC_Nacional'].isnull()]['fecha_mes'].unique()
    for fecha in sorted(fechas_sin_ipc)[:5]:
        count = len(df_enriquecido[(df_enriquecido['fecha_mes'] == fecha) & (df_enriquecido['IPC_Nacional'].isnull())])
        print(f"   • {fecha.strftime('%Y-%m')}: {count:,} registros")
else:
    print("\n✅ Todos los registros tienen IPC Nacional")

print("\n✅ INTEGRACIÓN IPC NACIONAL COMPLETADA")

🔗 INTEGRACIÓN 1: IPC NACIONAL

📊 ANTES DE LA INTEGRACIÓN:
   • Registros dataset principal: 99,457
   • Columnas: 18

📊 DESPUÉS DE LA INTEGRACIÓN:
   • Registros: 99,457
   • Columnas: 19
   • Nueva columna: IPC_Nacional

🔍 VALIDACIÓN:
   • Registros con IPC Nacional: 99,457 (100.00%)
   • Registros sin IPC Nacional: 0 (0.00%)

✅ Todos los registros tienen IPC Nacional

✅ INTEGRACIÓN IPC NACIONAL COMPLETADA


## 🔗 CELDA 8: INTEGRACIÓN 2 - LEFT JOIN con IPC por Categorías

In [8]:
print("="*80)
print("🔗 INTEGRACIÓN 2: IPC POR CATEGORÍAS ECOICOP")
print("="*80)

print("\n📊 ANTES DE LA INTEGRACIÓN:")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")

# LEFT JOIN con IPC por Categorías
df_enriquecido = df_enriquecido.merge(
    df_ipc_categorias[['fecha_mes', 'codigo_ecoicop', 'IPC_Categoria']],
    on=['fecha_mes', 'codigo_ecoicop'],
    how='left'
)

print("\n📊 DESPUÉS DE LA INTEGRACIÓN:")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")
print(f"   • Nueva columna: IPC_Categoria")

# Validar integración
nulos_ipc_cat = df_enriquecido['IPC_Categoria'].isnull().sum()
print(f"\n🔍 VALIDACIÓN:")
print(f"   • Registros con IPC Categoría: {len(df_enriquecido) - nulos_ipc_cat:,} ({(1 - nulos_ipc_cat/len(df_enriquecido))*100:.2f}%)")
print(f"   • Registros sin IPC Categoría: {nulos_ipc_cat:,} ({(nulos_ipc_cat/len(df_enriquecido))*100:.2f}%)")

if nulos_ipc_cat > 0:
    print(f"\n⚠️ Hay {nulos_ipc_cat} registros sin IPC por categoría")
else:
    print("\n✅ Todos los registros tienen IPC por Categoría")

print("\n✅ INTEGRACIÓN IPC CATEGORÍAS COMPLETADA")

🔗 INTEGRACIÓN 2: IPC POR CATEGORÍAS ECOICOP

📊 ANTES DE LA INTEGRACIÓN:
   • Registros: 99,457
   • Columnas: 19

📊 DESPUÉS DE LA INTEGRACIÓN:
   • Registros: 99,457
   • Columnas: 20
   • Nueva columna: IPC_Categoria

🔍 VALIDACIÓN:
   • Registros con IPC Categoría: 99,457 (100.00%)
   • Registros sin IPC Categoría: 0 (0.00%)

✅ Todos los registros tienen IPC por Categoría

✅ INTEGRACIÓN IPC CATEGORÍAS COMPLETADA


## 🔗 CELDA 9: INTEGRACIÓN 3 - LEFT JOIN con IPC por CCAA

In [9]:
print("="*80)
print("🔗 INTEGRACIÓN 3: IPC POR COMUNIDADES AUTÓNOMAS")
print("="*80)

print("\n📊 ANTES DE LA INTEGRACIÓN:")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")

# Asegurar que codigo_ccaa sea string en ambos datasets
df_enriquecido['codigo_ccaa'] = df_enriquecido['codigo_ccaa'].astype(str).str.zfill(2)
df_ipc_ccaa['codigo_ccaa'] = df_ipc_ccaa['codigo_ccaa'].astype(str).str.zfill(2)

# LEFT JOIN con IPC por CCAA
df_enriquecido = df_enriquecido.merge(
    df_ipc_ccaa[['fecha_mes', 'codigo_ccaa', 'IPC_CCAA']],
    on=['fecha_mes', 'codigo_ccaa'],
    how='left'
)

print("\n📊 DESPUÉS DE LA INTEGRACIÓN:")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")
print(f"   • Nueva columna: IPC_CCAA")

# Validar integración
nulos_ipc_ccaa = df_enriquecido['IPC_CCAA'].isnull().sum()
print(f"\n🔍 VALIDACIÓN:")
print(f"   • Registros con IPC CCAA: {len(df_enriquecido) - nulos_ipc_ccaa:,} ({(1 - nulos_ipc_ccaa/len(df_enriquecido))*100:.2f}%)")
print(f"   • Registros sin IPC CCAA: {nulos_ipc_ccaa:,} ({(nulos_ipc_ccaa/len(df_enriquecido))*100:.2f}%)")

if nulos_ipc_ccaa > 0:
    print(f"\n⚠️ Hay {nulos_ipc_ccaa} registros sin IPC por CCAA")
else:
    print("\n✅ Todos los registros tienen IPC por CCAA")

print("\n✅ INTEGRACIÓN IPC CCAA COMPLETADA")

🔗 INTEGRACIÓN 3: IPC POR COMUNIDADES AUTÓNOMAS

📊 ANTES DE LA INTEGRACIÓN:
   • Registros: 99,457
   • Columnas: 20

📊 DESPUÉS DE LA INTEGRACIÓN:
   • Registros: 99,457
   • Columnas: 21
   • Nueva columna: IPC_CCAA

🔍 VALIDACIÓN:
   • Registros con IPC CCAA: 99,457 (100.00%)
   • Registros sin IPC CCAA: 0 (0.00%)

✅ Todos los registros tienen IPC por CCAA

✅ INTEGRACIÓN IPC CCAA COMPLETADA


## ✅ CELDA 10: Validación Completa del Dataset Enriquecido

In [10]:
print("="*80)
print("✅ VALIDACIÓN COMPLETA DEL DATASET ENRIQUECIDO")
print("="*80)

print("\n1️⃣ DIMENSIONES:")
print(f"   • Total registros: {len(df_enriquecido):,}")
print(f"   • Total columnas: {len(df_enriquecido.columns)}")
print(f"   • Memoria: {df_enriquecido.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n2️⃣ NUEVAS COLUMNAS AÑADIDAS:")
columnas_nuevas = ['ciudad_espana', 'ccaa', 'codigo_ccaa', 'codigo_ecoicop',
                   'grupo_ecoicop', 'year', 'month', 'fecha_mes',
                   'IPC_Nacional', 'IPC_Categoria', 'IPC_CCAA']
for col in columnas_nuevas:
    if col in df_enriquecido.columns:
        print(f"   ✅ {col}")

print("\n3️⃣ VALORES NULOS EN COLUMNAS CRÍTICAS:")
columnas_criticas = ['IPC_Nacional', 'IPC_Categoria', 'IPC_CCAA', 'ciudad_espana', 'codigo_ecoicop']
nulos = df_enriquecido[columnas_criticas].isnull().sum()
nulos_pct = (nulos / len(df_enriquecido) * 100).round(2)
nulos_df = pd.DataFrame({
    'Columna': nulos.index,
    'Nulos': nulos.values,
    'Porcentaje (%)': nulos_pct.values
})
print(nulos_df.to_string(index=False))

print("\n4️⃣ ESTADÍSTICAS DE VARIABLES IPC:")
print("\nIPC Nacional:")
print(df_enriquecido['IPC_Nacional'].describe())
print("\nIPC Categoría:")
print(df_enriquecido['IPC_Categoria'].describe())
print("\nIPC CCAA:")
print(df_enriquecido['IPC_CCAA'].describe())

print("\n5️⃣ MUESTRA DEL DATASET ENRIQUECIDO:")
display(df_enriquecido[['invoice_date', 'ciudad_espana', 'category', 'grupo_ecoicop',
                        'price', 'IPC_Nacional', 'IPC_Categoria', 'IPC_CCAA']].head(10))

print("\n✅ VALIDACIÓN COMPLETADA")

✅ VALIDACIÓN COMPLETA DEL DATASET ENRIQUECIDO

1️⃣ DIMENSIONES:
   • Total registros: 99,457
   • Total columnas: 21
   • Memoria: 64.05 MB

2️⃣ NUEVAS COLUMNAS AÑADIDAS:
   ✅ ciudad_espana
   ✅ ccaa
   ✅ codigo_ccaa
   ✅ codigo_ecoicop
   ✅ grupo_ecoicop
   ✅ year
   ✅ month
   ✅ fecha_mes
   ✅ IPC_Nacional
   ✅ IPC_Categoria
   ✅ IPC_CCAA

3️⃣ VALORES NULOS EN COLUMNAS CRÍTICAS:
       Columna  Nulos  Porcentaje (%)
  IPC_Nacional      0             0.0
 IPC_Categoria      0             0.0
      IPC_CCAA      0             0.0
 ciudad_espana      0             0.0
codigo_ecoicop      0             0.0

4️⃣ ESTADÍSTICAS DE VARIABLES IPC:

IPC Nacional:
count    99457.000000
mean       105.258482
std          4.741450
min         97.008000
25%        100.046000
50%        107.375000
75%        109.866000
max        111.773000
Name: IPC_Nacional, dtype: float64

IPC Categoría:
count    99457.000000
mean       102.642101
std          6.148208
min         90.482000
25%         99.493000


,invoice_date,ciudad_espana,category,grupo_ecoicop,price,IPC_Nacional,IPC_Categoria,IPC_CCAA
0,2022-08-05,Barcelona,Clothing,Vestido y calzado,1500.40,109.498,99.711,109.050
1,2021-12-12,Alicante,Shoes,Vestido y calzado,1800.51,103.567,95.143,103.571
2,2021-11-09,Valencia,Clothing,Vestido y calzado,300.08,103.965,109.391,104.092
3,2021-05-16,Sevilla,Shoes,Vestido y calzado,3000.85,100.046,105.264,100.070
4,2021-10-24,Barcelona,Books,Ocio y cultura,60.60,102.738,100.126,102.508
5,2022-05-24,Alicante,Clothing,Vestido y calzado,1500.40,110.267,107.796,110.288
6,2022-03-13,Bilbao,Cosmetics,Otros bienes y servicios,40.66,107.375,102.859,106.959
7,2021-01-13,Madrid,Clothing,Vestido y calzado,600.16,97.008,90.482,97.330
8,2021-11-04,Valencia,Clothing,Vestido y calzado,900.24,103.965,109.391,104.092
9,2021-08-22,Barcelona,Clothing,Vestido y calzado,600.16,100.575,95.998,100.474



✅ VALIDACIÓN COMPLETADA


## 💾 CELDA 11: Guardar Dataset Enriquecido

In [11]:
print("="*80)
print("💾 GUARDANDO DATASET ENRIQUECIDO")
print("="*80)

# Crear directorio si no existe
import os
os.makedirs(ENRICHED_PATH, exist_ok=True)

# Guardar dataset enriquecido
ruta_guardado = ENRICHED_PATH + 'dataset_enriquecido_final.csv'
df_enriquecido.to_csv(ruta_guardado, index=False)

print(f"\n✅ Dataset enriquecido guardado en:")
print(f"   {ruta_guardado}")

# Verificar archivo guardado
file_size = os.path.getsize(ruta_guardado) / 1024**2
print(f"\n📊 INFORMACIÓN DEL ARCHIVO:")
print(f"   • Tamaño: {file_size:.2f} MB")
print(f"   • Registros: {len(df_enriquecido):,}")
print(f"   • Columnas: {len(df_enriquecido.columns)}")

print("\n✅ GUARDADO COMPLETADO")

💾 GUARDANDO DATASET ENRIQUECIDO

✅ Dataset enriquecido guardado en:
   /content/drive/MyDrive/Reto-6-Open-Data/data/enriched/dataset_enriquecido_final.csv

📊 INFORMACIÓN DEL ARCHIVO:
   • Tamaño: 15.67 MB
   • Registros: 99,457
   • Columnas: 21

✅ GUARDADO COMPLETADO


## 📝 CELDA 12: Resumen Final de la Integración

In [12]:
print("="*80)
print("📝 RESUMEN EJECUTIVO - INTEGRACIÓN DE DATASETS")
print("="*80)

print("\n✅ INTEGRACIÓN COMPLETADA EXITOSAMENTE")

print("\n📊 TRANSFORMACIONES APLICADAS:")
print("   1. ✅ Mapeo geográfico: Istanbul → 10 ciudades españolas")
print("   2. ✅ Mapeo categorías: 8 categorías → 5 grupos ECOICOP")
print("   3. ✅ Integración IPC Nacional (índice general)")
print("   4. ✅ Integración IPC por Categorías ECOICOP")
print("   5. ✅ Integración IPC por Comunidades Autónomas")

print("\n📈 DATASET ENRIQUECIDO FINAL:")
print(f"   • Registros totales: {len(df_enriquecido):,}")
print(f"   • Columnas originales: {len(df_principal.columns)}")
print(f"   • Columnas finales: {len(df_enriquecido.columns)}")
print(f"   • Columnas añadidas: {len(df_enriquecido.columns) - len(df_principal.columns)}")

print("\n🎯 MÉTRICAS DE CALIDAD:")
tasa_cobertura_nacional = ((len(df_enriquecido) - df_enriquecido['IPC_Nacional'].isnull().sum()) / len(df_enriquecido) * 100)
tasa_cobertura_categoria = ((len(df_enriquecido) - df_enriquecido['IPC_Categoria'].isnull().sum()) / len(df_enriquecido) * 100)
tasa_cobertura_ccaa = ((len(df_enriquecido) - df_enriquecido['IPC_CCAA'].isnull().sum()) / len(df_enriquecido) * 100)

print(f"   • Cobertura IPC Nacional: {tasa_cobertura_nacional:.2f}%")
print(f"   • Cobertura IPC Categoría: {tasa_cobertura_categoria:.2f}%")
print(f"   • Cobertura IPC CCAA: {tasa_cobertura_ccaa:.2f}%")

print("\n💾 ARCHIVO GENERADO:")
print("   • dataset_enriquecido_final.csv")

print("\n🎯 PRÓXIMOS PASOS:")
print("   1. ✅ Dataset principal + Open Data integrados")
print("   2. ⏳ Análisis comparativo (antes vs después)")
print("   3. ⏳ Cálculo de métricas enriquecidas")
print("   4. ⏳ Visualizaciones comparativas")
print("   5. ⏳ Informe final y conclusiones")

print("\n" + "="*80)
print("🎉 INTEGRACIÓN COMPLETADA - NOTEBOOK 03 FINALIZADO")
print("="*80)

📝 RESUMEN EJECUTIVO - INTEGRACIÓN DE DATASETS

✅ INTEGRACIÓN COMPLETADA EXITOSAMENTE

📊 TRANSFORMACIONES APLICADAS:
   1. ✅ Mapeo geográfico: Istanbul → 10 ciudades españolas
   2. ✅ Mapeo categorías: 8 categorías → 5 grupos ECOICOP
   3. ✅ Integración IPC Nacional (índice general)
   4. ✅ Integración IPC por Categorías ECOICOP
   5. ✅ Integración IPC por Comunidades Autónomas

📈 DATASET ENRIQUECIDO FINAL:
   • Registros totales: 99,457
   • Columnas originales: 10
   • Columnas finales: 21
   • Columnas añadidas: 11

🎯 MÉTRICAS DE CALIDAD:
   • Cobertura IPC Nacional: 100.00%
   • Cobertura IPC Categoría: 100.00%
   • Cobertura IPC CCAA: 100.00%

💾 ARCHIVO GENERADO:
   • dataset_enriquecido_final.csv

🎯 PRÓXIMOS PASOS:
   1. ✅ Dataset principal + Open Data integrados
   2. ⏳ Análisis comparativo (antes vs después)
   3. ⏳ Cálculo de métricas enriquecidas
   4. ⏳ Visualizaciones comparativas
   5. ⏳ Informe final y conclusiones

🎉 INTEGRACIÓN COMPLETADA - NOTEBOOK 03 FINALIZADO


---

## 📌 NOTAS FINALES:

### ✅ Lo que hemos logrado:
- **Enriquecido** el dataset principal de 99,457 transacciones con Open Data oficial del INE
- **Aplicado** 2 mapeos críticos (geográfico + categorías)
- **Integrado** 3 fuentes de IPC mediante LEFT JOIN
- **Validado** calidad de datos post-integración
- **Generado** dataset final enterprise-grade listo para análisis

### 🎯 Diferenciadores técnicos:
- Integración multi-fuente con validación rigurosa
- Mapeos personalizados Istanbul → España
- Cobertura temporal 2021-2023
- Sin pérdida de registros en LEFT JOIN
- Metodología reproducible y documentada

### 📊 Dataset Enriquecido Final:
- **99,457 registros** (sin pérdida)
- **11 nuevas columnas** de Open Data y transformaciones
- **3 indicadores IPC** (Nacional, Categoría, CCAA)
- **Listo para análisis comparativo**

---

**Autor:** Ana BM  
**Proyecto:** RETO 6 - Open Data - FUNDAE Business Analytics Nivel 4  
**GitHub:** https://github.com/AnaBMo/BA_Fundae_Nivel-4-BigData-ETL-DataMining  